In [ ]:

import os
import warnings
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
tf.random.set_seed(42)
np.random.seed(42)


TRAIN_PATH    = "Train_data.csv"
VAL_PATH      = "Validation_data.csv"
TEST_PATH     = "Test_data.csv"
OUTPUT_DIR    = "sota_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

LOOKBACK      = 168      
HORIZON       = 24       
LSTM_UNITS    = 128
DROPOUT_RATE  = 0.4
DENSE_UNITS   = 64
ATTN_UNITS    = 64
BATCH_SIZE    = 64
MAX_EPOCHS    = 50
LEARNING_RATE = 0.001

TARGETS = {
    "PM2.5": "pm25",
    "NO2":   "no2",
    "CO":    "co",
    "Ozone": "ozone",
}

POLLUTANT_FEATURES = ["pm10", "no", "nh3", "nox", "so2"]
METEO_FEATURES     = ["bp", "wind_speed", "air_temp", "humidity", "rainfall"]
CYCLICAL_FEATURES  = ["hour_sin", "hour_cos", "dow_sin", "dow_cos"]
ALL_FEATURES       = POLLUTANT_FEATURES + METEO_FEATURES + CYCLICAL_FEATURES




def load_dataframe(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, parse_dates=["from_date"])
    df = df.sort_values("from_date").reset_index(drop=True)

    df["hour"]     = df["from_date"].dt.hour
    df["dow"]      = df["from_date"].dt.dayofweek
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    df["dow_sin"]  = np.sin(2 * np.pi * df["dow"]  / 7)
    df["dow_cos"]  = np.cos(2 * np.pi * df["dow"]  / 7)

    df = df.fillna(method="ffill").fillna(method="bfill")
    return df


def build_arrays_for_target(train_df, val_df, test_df, target_col):
    other_targets = [col for col in TARGETS.values() if col != target_col]
    feature_cols  = ALL_FEATURES + other_targets

    train_df = train_df.dropna(subset=[target_col])
    val_df   = val_df.dropna(subset=[target_col])
    test_df  = test_df.dropna(subset=[target_col])

    X_tr_raw = train_df[feature_cols].values.astype(np.float32)
    X_va_raw = val_df[feature_cols].values.astype(np.float32)
    X_te_raw = test_df[feature_cols].values.astype(np.float32)

    y_tr = train_df[target_col].values.astype(np.float32)
    y_va = val_df[target_col].values.astype(np.float32)
    y_te = test_df[target_col].values.astype(np.float32)

    feat_scaler = RobustScaler()
    X_tr = feat_scaler.fit_transform(X_tr_raw)
    X_va = feat_scaler.transform(X_va_raw)
    X_te = feat_scaler.transform(X_te_raw)

    tgt_scaler = RobustScaler()
    y_tr_scaled = tgt_scaler.fit_transform(y_tr.reshape(-1, 1)).ravel()
    y_va_scaled = tgt_scaler.transform(y_va.reshape(-1, 1)).ravel()

    return (X_tr, y_tr_scaled,
            X_va, y_va_scaled,
            X_te, y_te,
            feat_scaler, tgt_scaler,
            feature_cols)


def make_sequences(X: np.ndarray, y: np.ndarray,
                   lookback: int = LOOKBACK, horizon: int = HORIZON):

    Xs, ys = [], []
    for i in range(lookback, len(X) - horizon + 1):
        Xs.append(X[i - lookback: i])
        ys.append(y[i: i + horizon])          
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32)




class BahdanauAttention(layers.Layer):
    def __init__(self, units: int = 64, **kwargs):
        super().__init__(**kwargs)
        self.W = layers.Dense(units, use_bias=False)
        self.V = layers.Dense(1,     use_bias=False)

    def call(self, hidden_states):
        score   = self.V(tf.nn.tanh(self.W(hidden_states)))    # (B, T, 1)
        alpha   = tf.nn.softmax(score, axis=1)                  # (B, T, 1)
        context = tf.reduce_sum(alpha * hidden_states, axis=1)  # (B, 2*units)
        return context, tf.squeeze(alpha, -1)

    def get_config(self):
        return super().get_config()




def build_model(
    timesteps:    int,
    n_features:   int,
    lstm_units:   int   = LSTM_UNITS,
    dropout_rate: float = DROPOUT_RATE,
    dense_units:  int   = DENSE_UNITS,
    attn_units:   int   = ATTN_UNITS,
    lr:           float = LEARNING_RATE,
    target_name:  str   = "pollutant",
    horizon:      int   = HORIZON,          
) -> Model:

    inp = keras.Input(shape=(timesteps, n_features), name="input")

    x = layers.Bidirectional(
        layers.LSTM(lstm_units, return_sequences=True), name="bilstm_1"
    )(inp)
    x = layers.Dropout(dropout_rate, name="dropout_1")(x)

    x = layers.Bidirectional(
        layers.LSTM(lstm_units, return_sequences=True), name="bilstm_2"
    )(x)
    x = layers.Dropout(dropout_rate, name="dropout_2")(x)

    context, _ = BahdanauAttention(units=attn_units, name="attention")(x)

    x   = layers.Dense(dense_units, activation="relu", name="dense_1")(context)
    # ← CHANGED: output horizon steps instead of 1
    out = layers.Dense(horizon, activation="linear", name="output")(x)

    model = Model(inputs=inp, outputs=out,
                  name=f"BiLSTM_Attn_{target_name}")
    model.compile(
        optimizer=keras.optimizers.RMSprop(learning_rate=lr),
        loss="mse",
        metrics=["mae"],
    )
    return model




def train_single_model(model, X_tr, y_tr, X_va, y_va, target_name):
    ckpt_path = os.path.join(OUTPUT_DIR, f"best_{target_name}.keras")
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=5,
            restore_best_weights=True, verbose=0
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=3, verbose=0
        ),
        keras.callbacks.ModelCheckpoint(
            filepath=ckpt_path, monitor="val_loss",
            save_best_only=True, verbose=0
        ),
    ]
    history = model.fit(
        X_tr, y_tr,
        validation_data=(X_va, y_va),
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        shuffle=False,
        verbose=1,
    )
    return history



def evaluate_model(model, X_te, y_te_raw, tgt_scaler, target_name):
    
    y_pred_scaled = model.predict(X_te, verbose=0)          # (N, 24)

    
    y_pred = tgt_scaler.inverse_transform(
        y_pred_scaled.reshape(-1, 1)
    ).reshape(y_pred_scaled.shape)                           # (N, 24)

    
    step_metrics = []
    for h in range(HORIZON):
        rmse = np.sqrt(mean_squared_error(y_te_raw[:, h], y_pred[:, h]))
        mae  = mean_absolute_error(y_te_raw[:, h], y_pred[:, h])
        r2   = r2_score(y_te_raw[:, h], y_pred[:, h])
        step_metrics.append({"step": h + 1, "RMSE": rmse, "MAE": mae, "R2": r2})

   
    avg_rmse = np.mean([m["RMSE"] for m in step_metrics])
    avg_mae  = np.mean([m["MAE"]  for m in step_metrics])
    avg_r2   = np.mean([m["R2"]   for m in step_metrics])

    print(f"  {target_name:<8}  Avg RMSE={avg_rmse:.4f}  "
          f"Avg MAE={avg_mae:.4f}  Avg R²={avg_r2:.4f}")

    
    step_df = pd.DataFrame(step_metrics)
    step_df["target"] = target_name
    step_csv = os.path.join(OUTPUT_DIR, f"step_metrics_{target_name}.csv")
    step_df.to_csv(step_csv, index=False)

    return {
        "target":    target_name,
        "RMSE":      avg_rmse,
        "MAE":       avg_mae,
        "R2":        avg_r2,
        "step_metrics": step_metrics,
        "y_true":    y_te_raw,   # (N, 24)
        "y_pred":    y_pred,     # (N, 24)
    }



def plot_results(all_results: list, all_histories: dict):
    palette = {"PM2.5": "#2563eb", "NO2": "#16a34a",
               "CO":    "#dc2626", "Ozone": "#ca8a04"}

    for res in all_results:
        name   = res["target"]
        hist   = all_histories[name]
        color  = palette[name]

        
        y_true_flat = res["y_true"].ravel()
        y_pred_flat = res["y_pred"].ravel()

        
        steps      = [m["step"] for m in res["step_metrics"]]
        step_rmses = [m["RMSE"] for m in res["step_metrics"]]

        fig, axes = plt.subplots(1, 4, figsize=(22, 4))
        fig.suptitle(
            f"SOTA Bi-LSTM+Attention — {name} (Delhi) | "
            f"Lookback={LOOKBACK}h → Horizon={HORIZON}h\n"
            f"Avg RMSE={res['RMSE']:.3f}  Avg MAE={res['MAE']:.3f}  "
            f"Avg R²={res['R2']:.3f}",
            fontsize=11, fontweight="bold"
        )

  
        ax = axes[0]
        ax.plot(hist.history["loss"],     color=color, label="Train")
        ax.plot(hist.history["val_loss"], color=color, label="Val",
                linestyle="--", alpha=0.6)
        ax.set_title("Loss (MSE)"); ax.set_xlabel("Epoch")
        ax.legend(); ax.grid(alpha=0.3)

       
        ax = axes[1]
        ax.plot(steps, step_rmses, color=color, marker="o", markersize=4)
        ax.set_title("RMSE per Forecast Step")
        ax.set_xlabel("Horizon (h)"); ax.set_ylabel("RMSE")
        ax.grid(alpha=0.3)

        ax = axes[2]
        n = min(500, len(y_true_flat))
        ax.plot(y_true_flat[:n], label="Actual",    color="black", linewidth=0.8)
        ax.plot(y_pred_flat[:n], label="Predicted", color=color,
                linewidth=0.8, linestyle="--", alpha=0.85)
        ax.set_title(f"Pred vs Actual (first {n} pts, flattened)")
        ax.set_xlabel("Point"); ax.set_ylabel(f"{name}")
        ax.legend(fontsize=8); ax.grid(alpha=0.3)


        ax = axes[3]
        ax.scatter(y_true_flat, y_pred_flat, alpha=0.10, s=4, color=color)
        lims = [min(y_true_flat.min(), y_pred_flat.min()),
                max(y_true_flat.max(), y_pred_flat.max())]
        ax.plot(lims, lims, "k--", linewidth=1, label="Perfect fit")
        ax.set_title("Scatter: Actual vs Predicted")
        ax.set_xlabel("Actual"); ax.set_ylabel("Predicted")
        ax.legend(fontsize=8); ax.grid(alpha=0.3)

        plt.tight_layout()
        save_path = os.path.join(OUTPUT_DIR, f"sota_{name.lower()}.png")
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.close()
        print(f"   Plot saved → {save_path}")


    names  = [r["target"] for r in all_results]
    rmses  = [r["RMSE"]   for r in all_results]
    maes   = [r["MAE"]    for r in all_results]
    r2s    = [r["R2"]     for r in all_results]
    colors = [palette[n]  for n in names]

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(
        f"SOTA Single-Task Bi-LSTM+Attention — Summary (Delhi) | "
        f"Lookback={LOOKBACK}h → Horizon={HORIZON}h",
        fontsize=12, fontweight="bold"
    )
    for ax, values, title, ylabel in zip(
        axes,
        [rmses, maes, r2s],
        ["Avg RMSE (lower is better)", "Avg MAE (lower is better)",
         "Avg R² (higher is better)"],
        ["RMSE", "MAE", "R²"],
    ):
        bars = ax.bar(names, values, color=colors, edgecolor="white")
        ax.set_title(title); ax.set_ylabel(ylabel)
        ax.grid(axis="y", alpha=0.3)
        for bar, v in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + max(values)*0.01,
                    f"{v:.3f}", ha="center", fontsize=9, fontweight="bold")

    plt.tight_layout()
    summary_path = os.path.join(OUTPUT_DIR, "sota_summary.png")
    plt.savefig(summary_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"   Summary plot saved → {summary_path}")



def main():
    print("─"*60)
    print(f"  SOTA: 4× Single-Task Stacked Bi-LSTM + Attention (Delhi)")
    print(f"  Lookback={LOOKBACK}h  →  Horizon={HORIZON}h")
    print("  Targets: PM2.5 | NO2 | CO | Ozone")
    print("─"*60)

    print("\n[1/3] Loading data …")
    train_df = load_dataframe(TRAIN_PATH)
    val_df   = load_dataframe(VAL_PATH)
    test_df  = load_dataframe(TEST_PATH)
    print(f"   Train: {len(train_df)} rows  |  "
          f"Val: {len(val_df)} rows  |  Test: {len(test_df)} rows")

    print("\n[2/3] Training single-task models …\n")
    all_results   = []
    all_histories = {}

    for display_name, col_name in TARGETS.items():
        print(f"\n{'='*50}")
        print(f"  Target: {display_name}  (column: {col_name})")
        print(f"{'='*50}")

        (X_tr, y_tr,
         X_va, y_va,
         X_te, y_te_raw,
         feat_scaler, tgt_scaler,
         feature_cols) = build_arrays_for_target(
            train_df, val_df, test_df, col_name
        )

        X_tr_seq, y_tr_seq = make_sequences(X_tr, y_tr)
        X_va_seq, y_va_seq = make_sequences(X_va, y_va)
        X_te_seq, y_te_seq = make_sequences(X_te, y_te_raw)

        print(f"  Sequences — train: {X_tr_seq.shape}  "
              f"val: {X_va_seq.shape}  test: {X_te_seq.shape}")
        print(f"  Target shape — train: {y_tr_seq.shape}")   # ← (N, 24)

        model = build_model(
            timesteps=LOOKBACK,
            n_features=X_tr_seq.shape[2],
            target_name=display_name,
        )

        history = train_single_model(
            model, X_tr_seq, y_tr_seq,
            X_va_seq, y_va_seq,
            display_name,
        )
        all_histories[display_name] = history

        print(f"\n  Test Results:")
        result = evaluate_model(
            model, X_te_seq, y_te_seq, tgt_scaler, display_name
        )
        all_results.append(result)

    print("\n" + "="*55)
    print("  FINAL SUMMARY — SOTA Single-Task Bi-LSTM+Attention")
    print(f"  Lookback={LOOKBACK}h  Horizon={HORIZON}h")
    print("="*55)
    print(f"  {'Pollutant':<10} {'Avg RMSE':>10} {'Avg MAE':>10} {'Avg R²':>10}")
    print("  " + "-"*42)
    for r in all_results:
        print(f"  {r['target']:<10} {r['RMSE']:>10.4f} "
              f"{r['MAE']:>10.4f} {r['R2']:>10.4f}")
    print("="*55)

    print("\n[3/3] Saving results …")
    metrics_df = pd.DataFrame([
        {"model": "SOTA_BiLSTM_Attention",
         "target": r["target"],
         "Avg_RMSE": round(r["RMSE"], 4),
         "Avg_MAE":  round(r["MAE"],  4),
         "Avg_R2":   round(r["R2"],   4)}
        for r in all_results
    ])
    csv_path = os.path.join(OUTPUT_DIR, "sota_metrics.csv")
    metrics_df.to_csv(csv_path, index=False)
    print(f"   Metrics saved → {csv_path}")

    plot_results(all_results, all_histories)
    print("\nDone ✓")
    return all_results, metrics_df


if __name__ == "__main__":
    all_results, metrics_df = main()

────────────────────────────────────────────────────────────
  SOTA: 4× Single-Task Stacked Bi-LSTM + Attention (Delhi)
  Lookback=168h  →  Horizon=24h
  Targets: PM2.5 | NO2 | CO | Ozone
────────────────────────────────────────────────────────────

[1/3] Loading data …
   Train: 52584 rows  |  Val: 1416 rows  |  Test: 744 rows

[2/3] Training single-task models …


  Target: PM2.5  (column: pm25)
  Sequences — train: (52393, 168, 17)  val: (1225, 168, 17)  test: (553, 168, 17)
  Target shape — train: (52393, 24)
Epoch 1/50
819/819 ━━━━━━━━━━━━━━━━━━━━ 37s 37ms/step - loss: 0.4187 - mae: 0.4318 - val_loss: 0.9073 - val_mae: 0.8159 - learning_rate: 0.0010
Epoch 2/50
819/819 ━━━━━━━━━━━━━━━━━━━━ 40s 38ms/step - loss: 0.3435 - mae: 0.3790 - val_loss: 0.6079 - val_mae: 0.6409 - learning_rate: 0.0010
Epoch 3/50
819/819 ━━━━━━━━━━━━━━━━━━━━ 31s 38ms/step - loss: 0.2619 - mae: 0.3157 - val_loss: 0.4926 - val_mae: 0.5541 - learning_rate: 0.0010
Epoch 4/50
819/819 ━━━━━━━━━━━━━━━━━━━━ 32s 39ms/

In [ ]:


import os
import warnings
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
tf.random.set_seed(42)
np.random.seed(42)


TRAIN_PATH    = "Train_data.csv"
VAL_PATH      = "Validation_data.csv"
TEST_PATH     = "Test_data.csv"
OUTPUT_DIR    = "mtl_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

LOOKBACK      = 168      
HORIZON       = 24       
LSTM_UNITS    = 128
DROPOUT_RATE  = 0.4
DENSE_UNITS   = 64
ATTN_UNITS    = 64
BATCH_SIZE    = 64
MAX_EPOCHS    = 50
LEARNING_RATE = 0.001
PATIENCE      = 5

TARGETS = {
    "PM2.5": "pm25",
    "NO2":   "no2",
    "CO":    "co",
    "Ozone": "ozone",
}
TARGET_KEYS = list(TARGETS.keys())
TARGET_COLS = list(TARGETS.values())
N_TASKS     = len(TARGET_KEYS)          # 4

POLLUTANT_FEATURES = ["pm10", "no", "nh3", "nox", "so2"]
METEO_FEATURES     = ["bp", "wind_speed", "air_temp", "humidity", "rainfall"]
CYCLICAL_FEATURES  = ["hour_sin", "hour_cos", "dow_sin", "dow_cos"]
ALL_FEATURES       = POLLUTANT_FEATURES + METEO_FEATURES + CYCLICAL_FEATURES

PALETTE = {"PM2.5": "#2563eb", "NO2": "#16a34a",
           "CO":    "#dc2626", "Ozone": "#ca8a04"}




def load_dataframe(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, parse_dates=["from_date"])
    df = df.sort_values("from_date").reset_index(drop=True)

    df["hour"]     = df["from_date"].dt.hour
    df["dow"]      = df["from_date"].dt.dayofweek
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    df["dow_sin"]  = np.sin(2 * np.pi * df["dow"]  / 7)
    df["dow_cos"]  = np.cos(2 * np.pi * df["dow"]  / 7)

    df = df.fillna(method="ffill").fillna(method="bfill")
    return df


def build_arrays(train_df, val_df, test_df):
    feature_cols = ALL_FEATURES + TARGET_COLS

    train_df = train_df.dropna(subset=TARGET_COLS)
    val_df   = val_df.dropna(subset=TARGET_COLS)
    test_df  = test_df.dropna(subset=TARGET_COLS)

    X_tr_raw = train_df[feature_cols].values.astype(np.float32)
    X_va_raw = val_df[feature_cols].values.astype(np.float32)
    X_te_raw = test_df[feature_cols].values.astype(np.float32)

    feat_scaler = RobustScaler()
    X_tr = feat_scaler.fit_transform(X_tr_raw)
    X_va = feat_scaler.transform(X_va_raw)
    X_te = feat_scaler.transform(X_te_raw)

    tgt_scalers  = {}
    Y_tr_scaled  = np.zeros((len(train_df), N_TASKS), dtype=np.float32)
    Y_va_scaled  = np.zeros((len(val_df),   N_TASKS), dtype=np.float32)
    Y_te_raw_arr = np.zeros((len(test_df),  N_TASKS), dtype=np.float32)

    for i, col in enumerate(TARGET_COLS):
        sc = RobustScaler()
        Y_tr_scaled[:, i]  = sc.fit_transform(
            train_df[col].values.reshape(-1, 1)
        ).ravel()
        Y_va_scaled[:, i]  = sc.transform(
            val_df[col].values.reshape(-1, 1)
        ).ravel()
        Y_te_raw_arr[:, i] = test_df[col].values
        tgt_scalers[col]   = sc

    return (X_tr, Y_tr_scaled,
            X_va, Y_va_scaled,
            X_te, Y_te_raw_arr,
            feat_scaler, tgt_scalers,
            feature_cols)


def make_sequences(X: np.ndarray, Y: np.ndarray,
                   lookback: int = LOOKBACK, horizon: int = HORIZON):

    Xs, Ys = [], []
    for i in range(lookback, len(X) - horizon + 1):
        Xs.append(X[i - lookback: i])
        # Stack 24-step windows for each of the 4 tasks → flatten to (96,)
        Ys.append(Y[i: i + horizon, :].T.ravel())  # (N_TASKS * horizon,) ← CHANGED
    return np.array(Xs, dtype=np.float32), np.array(Ys, dtype=np.float32)




class BahdanauAttention(layers.Layer):
    def __init__(self, units: int = 64, **kwargs):
        super().__init__(**kwargs)
        self.W = layers.Dense(units, use_bias=False)
        self.V = layers.Dense(1,     use_bias=False)

    def call(self, hidden_states):
        score   = self.V(tf.nn.tanh(self.W(hidden_states)))
        alpha   = tf.nn.softmax(score, axis=1)
        context = tf.reduce_sum(alpha * hidden_states, axis=1)
        return context, tf.squeeze(alpha, -1)

    def get_config(self):
        return super().get_config()




def build_mtl_model(
    timesteps:    int,
    n_features:   int,
    lstm_units:   int   = LSTM_UNITS,
    dropout_rate: float = DROPOUT_RATE,
    dense_units:  int   = DENSE_UNITS,
    attn_units:   int   = ATTN_UNITS,
    horizon:      int   = HORIZON,      # ← NEW
) -> Model:
    inp = keras.Input(shape=(timesteps, n_features), name="input")

   
    x = layers.Bidirectional(
        layers.LSTM(lstm_units, return_sequences=True),
        name="shared_bilstm_1"
    )(inp)
    x = layers.Dropout(dropout_rate, name="shared_dropout_1")(x)

    x = layers.Bidirectional(
        layers.LSTM(lstm_units, return_sequences=True),
        name="shared_bilstm_2"
    )(x)
    x = layers.Dropout(dropout_rate, name="shared_dropout_2")(x)

    context, _ = BahdanauAttention(
        units=attn_units, name="shared_attention"
    )(x)

    
    outputs = []
    for task_name in TARGET_KEYS:
        safe = task_name.replace(".", "_")
        h   = layers.Dense(
            dense_units, activation="relu",
            name=f"{safe}_dense"
        )(context)
        
        out = layers.Dense(
            horizon, activation="linear",
            name=f"{safe}_output"
        )(h)
        outputs.append(out)

    
    combined = layers.Concatenate(name="combined")(outputs)

    model = Model(inputs=inp, outputs=combined,
                  name="MTL_BiLSTM_Attention")
    model.compile(
        optimizer=keras.optimizers.RMSprop(learning_rate=LEARNING_RATE),
        loss="mse",
        metrics=["mae"],
    )
    return model




def train_model(model, X_tr, Y_tr, X_va, Y_va):
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=PATIENCE,
            restore_best_weights=True, verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=3, verbose=1
        ),
        keras.callbacks.ModelCheckpoint(
            filepath=os.path.join(OUTPUT_DIR, "best_mtl_model.keras"),
            monitor="val_loss", save_best_only=True, verbose=0
        ),
    ]
    history = model.fit(
        X_tr, Y_tr,
        validation_data=(X_va, Y_va),
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        shuffle=False,
        verbose=1,
    )
    return history



def evaluate_mtl(model, X_te, Y_te_raw_seq, tgt_scalers):
    Y_pred_flat = model.predict(X_te, verbose=0)          

    
    Y_pred_3d = Y_pred_flat.reshape(-1, N_TASKS, HORIZON)
    Y_true_3d = Y_te_raw_seq.reshape(-1, N_TASKS, HORIZON)  

    results = []
    print("\n" + "="*60)
    print("  MTL Test Results — Delhi  (Avg over 24-step horizon)")
    print("="*60)
    print(f"  {'Pollutant':<10} {'Avg RMSE':>10} {'Avg MAE':>10} {'Avg R²':>10}")
    print("  " + "-"*46)

    for i, (display_name, col_name) in enumerate(TARGETS.items()):
        
        pred_scaled = Y_pred_3d[:, i, :]                  
        pred_raw = tgt_scalers[col_name].inverse_transform(
            pred_scaled.reshape(-1, 1)
        ).reshape(-1, HORIZON)                               

        true_raw = Y_true_3d[:, i, :]                       

        step_metrics = []
        for h in range(HORIZON):
            rmse = np.sqrt(mean_squared_error(true_raw[:, h], pred_raw[:, h]))
            mae  = mean_absolute_error(true_raw[:, h], pred_raw[:, h])
            r2   = r2_score(true_raw[:, h], pred_raw[:, h])
            step_metrics.append({"step": h + 1, "RMSE": rmse, "MAE": mae, "R2": r2})

        avg_rmse = np.mean([m["RMSE"] for m in step_metrics])
        avg_mae  = np.mean([m["MAE"]  for m in step_metrics])
        avg_r2   = np.mean([m["R2"]   for m in step_metrics])

        print(f"  {display_name:<10} {avg_rmse:>10.4f} {avg_mae:>10.4f} {avg_r2:>10.4f}")

        
        step_df = pd.DataFrame(step_metrics)
        step_df["target"] = display_name
        step_df.to_csv(
            os.path.join(OUTPUT_DIR, f"step_metrics_{display_name}.csv"),
            index=False
        )

        results.append({
            "target":       display_name,
            "RMSE":         avg_rmse,
            "MAE":          avg_mae,
            "R2":           avg_r2,
            "step_metrics": step_metrics,
            "y_true":       true_raw,   # (N, 24)
            "y_pred":       pred_raw,   # (N, 24)
        })

    print("="*60)
    return results



def plot_training(history):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle(
        f"MTL Training — Stacked Bi-LSTM + Attention | "
        f"Lookback={LOOKBACK}h → Horizon={HORIZON}h",
        fontsize=12, fontweight="bold"
    )
    axes[0].plot(history.history["loss"],     label="Train", color="#1e3a5f")
    axes[0].plot(history.history["val_loss"], label="Val",
                 color="#1e3a5f", linestyle="--", alpha=0.7)
    axes[0].set_title("Total Loss (MSE)")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("MSE")
    axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(history.history["mae"],     label="Train MAE", color="#16a34a")
    axes[1].plot(history.history["val_mae"], label="Val MAE",
                 color="#16a34a", linestyle="--", alpha=0.7)
    axes[1].set_title("MAE")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("MAE")
    axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, "mtl_training.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"   Training plot → {path}")


def plot_predictions(results):
    for res in results:
        name  = res["target"]
        color = PALETTE[name]

        y_true_flat = res["y_true"].ravel()
        y_pred_flat = res["y_pred"].ravel()
        n = min(500, len(y_true_flat))

        steps      = [m["step"] for m in res["step_metrics"]]
        step_rmses = [m["RMSE"] for m in res["step_metrics"]]

        fig, axes = plt.subplots(1, 3, figsize=(18, 4))
        fig.suptitle(
            f"MTL — {name} (Delhi) | Lookback={LOOKBACK}h → Horizon={HORIZON}h\n"
            f"Avg RMSE={res['RMSE']:.3f}  Avg MAE={res['MAE']:.3f}  "
            f"Avg R²={res['R2']:.3f}",
            fontsize=11, fontweight="bold"
        )

      
        ax = axes[0]
        ax.plot(steps, step_rmses, color=color, marker="o", markersize=4)
        ax.set_title("RMSE per Forecast Step")
        ax.set_xlabel("Horizon (h)"); ax.set_ylabel("RMSE")
        ax.grid(alpha=0.3)

     
        ax = axes[1]
        ax.plot(y_true_flat[:n], label="Actual",    color="black", linewidth=0.8)
        ax.plot(y_pred_flat[:n], label="Predicted", color=color,
                linewidth=0.8, linestyle="--", alpha=0.85)
        ax.set_title(f"Pred vs Actual (first {n} pts, flattened)")
        ax.set_xlabel("Point"); ax.set_ylabel(f"{name}")
        ax.legend(fontsize=8); ax.grid(alpha=0.3)

        ax = axes[2]
        ax.scatter(y_true_flat, y_pred_flat, alpha=0.10, s=4, color=color)
        lims = [min(y_true_flat.min(), y_pred_flat.min()),
                max(y_true_flat.max(), y_pred_flat.max())]
        ax.plot(lims, lims, "k--", linewidth=1, label="Perfect fit")
        ax.set_title("Scatter: Actual vs Predicted")
        ax.set_xlabel("Actual"); ax.set_ylabel("Predicted")
        ax.legend(fontsize=8); ax.grid(alpha=0.3)

        plt.tight_layout()
        path = os.path.join(OUTPUT_DIR, f"mtl_{name.lower()}.png")
        plt.savefig(path, dpi=150, bbox_inches="tight")
        plt.close()
        print(f"   Plot → {path}")


def plot_comparison(mtl_results):
    """Side-by-side SOTA vs MTL. Runs only if sota_metrics.csv exists."""
    sota_csv = os.path.join("sota_outputs", "sota_metrics.csv")
    if not os.path.exists(sota_csv):
        print("   sota_metrics.csv not found — skipping comparison plot.")
        return

    sota_df = pd.read_csv(sota_csv)
    mtl_df  = pd.DataFrame([
        {"target": r["target"], "Avg_RMSE": r["RMSE"],
         "Avg_MAE": r["MAE"], "Avg_R2": r["R2"]}
        for r in mtl_results
    ])

 
    metrics      = ["Avg_RMSE", "Avg_MAE", "Avg_R2"]
    titles       = ["Avg RMSE  (↓ better)", "Avg MAE  (↓ better)",
                    "Avg R²  (↑ better)"]
    x, width     = np.arange(len(TARGET_KEYS)), 0.35

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(
        f"SOTA vs MTL — Delhi | Lookback={LOOKBACK}h → Horizon={HORIZON}h",
        fontsize=13, fontweight="bold"
    )

    for ax, metric, title in zip(axes, metrics, titles):
        sota_vals = [
            float(sota_df.loc[sota_df["target"] == t, metric].values[0])
            for t in TARGET_KEYS
        ]
        mtl_vals = [
            float(mtl_df.loc[mtl_df["target"] == t, metric].values[0])
            for t in TARGET_KEYS
        ]
        all_vals = sota_vals + mtl_vals

        b1 = ax.bar(x - width/2, sota_vals, width,
                    label="SOTA (Single-Task)", color="#94a3b8", edgecolor="white")
        b2 = ax.bar(x + width/2, mtl_vals,   width,
                    label="MTL (Ours)",          color="#2563eb", edgecolor="white")

        ax.set_title(title); ax.set_xticks(x)
        ax.set_xticklabels(TARGET_KEYS); ax.legend(fontsize=8)
        ax.grid(axis="y", alpha=0.3)

        for bar, v in zip(list(b1) + list(b2), all_vals):
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + max(all_vals) * 0.01,
                    f"{v:.3f}", ha="center", fontsize=7, fontweight="bold")

    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, "sota_vs_mtl_comparison.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"   Comparison plot → {path}")


def main():
    print("─"*60)
    print("  MTL: Stacked Bi-LSTM + Attention — Delhi")
    print(f"  Lookback={LOOKBACK}h  →  Horizon={HORIZON}h")
    print("  Tasks: PM2.5 | NO2 | CO | Ozone")
    print("─"*60)

    print("\n[1/5] Loading data …")
    train_df = load_dataframe(TRAIN_PATH)
    val_df   = load_dataframe(VAL_PATH)
    test_df  = load_dataframe(TEST_PATH)
    print(f"   Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}")

    print("\n[2/5] Preprocessing …")
    (X_tr, Y_tr,
     X_va, Y_va,
     X_te, Y_te_raw,
     feat_scaler, tgt_scalers,
     feature_cols) = build_arrays(train_df, val_df, test_df)

    X_tr_seq, Y_tr_seq = make_sequences(X_tr, Y_tr)
    X_va_seq, Y_va_seq = make_sequences(X_va, Y_va)
    X_te_seq, Y_te_seq = make_sequences(X_te, Y_te_raw)

    print(f"   Input features ({len(feature_cols)}): {feature_cols}")
    print(f"   X_train: {X_tr_seq.shape}   Y_train: {Y_tr_seq.shape}")  # (N, 96)
    print(f"   X_val  : {X_va_seq.shape}   Y_val  : {Y_va_seq.shape}")
    print(f"   X_test : {X_te_seq.shape}   Y_test : {Y_te_seq.shape}")

    print("\n[3/5] Building model …")
    model = build_mtl_model(
        timesteps=LOOKBACK,
        n_features=X_tr_seq.shape[2],
    )
    model.summary()
    print(f"\n   Total parameters: {model.count_params():,}")

    print("\n[4/5] Training …")
    history = train_model(model, X_tr_seq, Y_tr_seq,
                          X_va_seq, Y_va_seq)

    print("\n[5/5] Evaluating …")
    mtl_results = evaluate_mtl(model, X_te_seq, Y_te_seq, tgt_scalers)

    metrics_df = pd.DataFrame([
        {"model":    "MTL_BiLSTM_Attention",
         "target":   r["target"],
         "Avg_RMSE": round(r["RMSE"], 4),
         "Avg_MAE":  round(r["MAE"],  4),
         "Avg_R2":   round(r["R2"],   4)}
        for r in mtl_results
    ])
    csv_path = os.path.join(OUTPUT_DIR, "mtl_metrics.csv")
    metrics_df.to_csv(csv_path, index=False)
    print(f"\n   Metrics saved → {csv_path}")

    print("\n   Generating plots …")
    plot_training(history)
    plot_predictions(mtl_results)
    plot_comparison(mtl_results)

    print("\nDone ✓")
    return model, mtl_results, metrics_df


if __name__ == "__main__":
    model, mtl_results, metrics_df = main()

────────────────────────────────────────────────────────────
  MTL: Stacked Bi-LSTM + Attention — Delhi
  Lookback=168h  →  Horizon=24h
  Tasks: PM2.5 | NO2 | CO | Ozone
────────────────────────────────────────────────────────────

[1/5] Loading data …
   Train: 52584  Val: 1416  Test: 744

[2/5] Preprocessing …
   Input features (18): ['pm10', 'no', 'nh3', 'nox', 'so2', 'bp', 'wind_speed', 'air_temp', 'humidity', 'rainfall', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'pm25', 'no2', 'co', 'ozone']
   X_train: (52393, 168, 18)   Y_train: (52393, 96)
   X_val  : (1225, 168, 18)   Y_val  : (1225, 96)
   X_test : (553, 168, 18)   Y_test : (553, 96)

[3/5] Building model …


Model: "MTL_BiLSTM_Attention"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 168, 18)   │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_bilstm_1     │ (None, 168, 256)  │    150,528 │ input[0][0]       │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_dropout_1    │ (None, 168, 256)  │          0 │ shared_bilstm_1[… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_bilstm_2     │ (None, 168, 256)  │    394,240 │ shared_dropout_1… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_dropout_2    │ (None, 168, 256)  │          0 │ shared_bilstm_2[… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_attention    │ [(None, 256),     │     16,448 │ shared_dropout_2… │
│ (BahdanauAttention) │ (None, 168)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ PM2_5_dense (Dense) │ (None, 64)        │     16,448 │ shared_attention… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ NO2_dense (Dense)   │ (None, 64)        │     16,448 │ shared_attention… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ CO_dense (Dense)    │ (None, 64)        │     16,448 │ shared_attention… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Ozone_dense (Dense) │ (None, 64)        │     16,448 │ shared_attention… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ PM2_5_output        │ (None, 24)        │      1,560 │ PM2_5_dense[0][0] │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ NO2_output (Dense)  │ (None, 24)        │      1,560 │ NO2_dense[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ CO_output (Dense)   │ (None, 24)        │      1,560 │ CO_dense[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Ozone_output        │ (None, 24)        │      1,560 │ Ozone_dense[0][0] │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ combined            │ (None, 96)        │          0 │ PM2_5_output[0][… │
│ (Concatenate)       │                   │            │ NO2_output[0][0], │
│                     │                   │            │ CO_output[0][0],  │
│                     │                   │            │ Ozone_output[0][… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 633,248 (2.42 MB)

 Trainable params: 633,248 (2.42 MB)

 Non-trainable params: 0 (0.00 B)


   Total parameters: 633,248

[4/5] Training …
Epoch 1/50
819/819 ━━━━━━━━━━━━━━━━━━━━ 35s 37ms/step - loss: 0.4764 - mae: 0.4613 - val_loss: 0.3579 - val_mae: 0.4420 - learning_rate: 0.0010
Epoch 2/50
819/819 ━━━━━━━━━━━━━━━━━━━━ 30s 36ms/step - loss: 0.2931 - mae: 0.3437 - val_loss: 0.3035 - val_mae: 0.3962 - learning_rate: 0.0010
Epoch 3/50
819/819 ━━━━━━━━━━━━━━━━━━━━ 31s 37ms/step - loss: 0.2729 - mae: 0.3283 - val_loss: 0.2893 - val_mae: 0.3813 - learning_rate: 0.0010
Epoch 4/50
819/819 ━━━━━━━━━━━━━━━━━━━━ 31s 37ms/step - loss: 0.2638 - mae: 0.3207 - val_loss: 0.2805 - val_mae: 0.3717 - learning_rate: 0.0010
Epoch 5/50
819/819 ━━━━━━━━━━━━━━━━━━━━ 31s 38ms/step - loss: 0.2561 - mae: 0.3149 - val_loss: 0.2772 - val_mae: 0.3684 - learning_rate: 0.0010
Epoch 6/50
819/819 ━━━━━━━━━━━━━━━━━━━━ 31s 38ms/step - loss: 0.2504 - mae: 0.3106 - val_loss: 0.2682 - val_mae: 0.3609 - learning_rate: 0.0010
Epoch 7/50
819/819 ━━━━━━━━━━━━━━━━━━━━ 31s 38ms/step - loss: 0.2453 - mae: 0.3069 - val

In [ ]:
!zip -r sota_outputs.zip /content/sota_outputs

  adding: content/sota_outputs/ (stored 0%)
  adding: content/sota_outputs/sota_metrics.csv (deflated 39%)
  adding: content/sota_outputs/sota_co.png (deflated 4%)
  adding: content/sota_outputs/step_metrics_CO.csv (deflated 50%)
  adding: content/sota_outputs/sota_summary.png (deflated 18%)
  adding: content/sota_outputs/best_NO2.keras (deflated 9%)
  adding: content/sota_outputs/best_PM2.5.keras (deflated 9%)
  adding: content/sota_outputs/sota_no2.png (deflated 4%)
  adding: content/sota_outputs/sota_ozone.png (deflated 3%)
  adding: content/sota_outputs/step_metrics_PM2.5.csv (deflated 50%)
  adding: content/sota_outputs/sota_pm2.5.png (deflated 4%)
  adding: content/sota_outputs/best_CO.keras (deflated 9%)
  adding: content/sota_outputs/step_metrics_NO2.csv (deflated 50%)
  adding: content/sota_outputs/best_Ozone.keras (deflated 9%)
  adding: content/sota_outputs/step_metrics_Ozone.csv (deflated 51%)


In [ ]:
!zip -r mtl_outputs.zip /content/mtl_outputs

  adding: content/mtl_outputs/ (stored 0%)
  adding: content/mtl_outputs/mtl_training.png (deflated 10%)
  adding: content/mtl_outputs/mtl_co.png (deflated 4%)
  adding: content/mtl_outputs/step_metrics_CO.csv (deflated 51%)
  adding: content/mtl_outputs/best_mtl_model.keras (deflated 9%)
  adding: content/mtl_outputs/step_metrics_PM2.5.csv (deflated 50%)
  adding: content/mtl_outputs/mtl_metrics.csv (deflated 38%)
  adding: content/mtl_outputs/step_metrics_NO2.csv (deflated 49%)
  adding: content/mtl_outputs/mtl_no2.png (deflated 3%)
  adding: content/mtl_outputs/step_metrics_Ozone.csv (deflated 51%)
  adding: content/mtl_outputs/sota_vs_mtl_comparison.png (deflated 19%)
  adding: content/mtl_outputs/mtl_ozone.png (deflated 3%)
  adding: content/mtl_outputs/mtl_pm2.5.png (deflated 3%)
